In [86]:
import math
import random
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# 1. CONFIGURATION
# ============================================================

# Folder containing all CSV files
CSV_FOLDER = "."

CHECKPOINT_PATH = "csv_llm_1M_checkpoint.pth"
WEIGHTS_PATH = "csv_llm_1M_weights.pth"

SEED = 42

# Training settings
batch_size = 16
block_size = 128
training_steps = 2000
eval_interval = 100
eval_iters = 20

learning_rate = 3e-4
weight_decay = 0.01
dropout = 0.1

# Approximately 1M parameter model
embedding_dim = 128
num_heads = 4
num_layers = 5


# ============================================================
# 2. DEVICE
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)


# ============================================================
# 3. RANDOM SEED
# ============================================================

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 4. FIND ALL CSV FILES
# ============================================================

csv_files = sorted(
    Path(CSV_FOLDER).glob("*.csv")
)

if not csv_files:
    raise FileNotFoundError(
        f"No CSV files found in folder: {CSV_FOLDER}"
    )


print("\n====================================")
print("CSV FILES FOUND")
print("====================================")

for file in csv_files:
    print(file.name)

print("\nTotal CSV files:", len(csv_files))


# ============================================================
# 5. CHECK COLUMN CONSISTENCY
# ============================================================

reference_file = csv_files[0]

reference_columns = list(
    pd.read_csv(
        reference_file,
        nrows=0
    ).columns
)

reference_column_set = set(
    reference_columns
)


print("\n====================================")
print("COLUMN CHECK")
print("====================================")

print(
    f"Reference file: {reference_file.name}"
)

print(
    f"Reference columns: "
    f"{len(reference_columns)}"
)


all_columns_match = True


for file in csv_files:

    columns = list(
        pd.read_csv(
            file,
            nrows=0
        ).columns
    )

    column_set = set(columns)

    missing_columns = (
        reference_column_set
        - column_set
    )

    extra_columns = (
        column_set
        - reference_column_set
    )

    print(
        f"\n{file.name}: "
        f"{len(columns)} columns"
    )

    if not missing_columns and not extra_columns:

        print("  Columns match")

    else:

        all_columns_match = False

        print("  Column mismatch")

        if missing_columns:
            print(
                "  Missing columns:",
                sorted(missing_columns)
            )

        if extra_columns:
            print(
                "  Additional columns:",
                sorted(extra_columns)
            )


# ============================================================
# 6. STOP IF COLUMN STRUCTURE IS DIFFERENT
# ============================================================

if not all_columns_match:

    raise ValueError(
        "\nSome CSV files have different columns. "
        "Fix the column mismatch before training."
    )


print(
    "\nAll CSV files have compatible columns."
)


# ============================================================
# 7. READ AND MERGE ALL CSV FILES
# ============================================================

dataframes = []

print("\n====================================")
print("READING CSV FILES")
print("====================================")


for file in csv_files:

    temp_df = pd.read_csv(file)

    # Ensure same column order as first CSV
    temp_df = temp_df[
        reference_columns
    ]

    print(
        f"{file.name}: "
        f"{len(temp_df)} rows, "
        f"{len(temp_df.columns)} columns"
    )

    dataframes.append(
        temp_df
    )


df = pd.concat(
    dataframes,
    ignore_index=True
)


print("\n====================================")
print("MERGED DATASET")
print("====================================")

print(
    "Total CSV files:",
    len(csv_files)
)

print(
    "Total rows:",
    len(df)
)

print(
    "Total columns:",
    len(df.columns)
)


print("\nColumn names:")

for col in df.columns:
    print(" -", col)


# ============================================================
# 8. OPTIONAL: SAVE MERGED CSV
# ============================================================

MERGED_CSV_PATH = "merged_training_data.csv"

df.to_csv(
    MERGED_CSV_PATH,
    index=False
)

print(
    "\nMerged CSV saved as:",
    MERGED_CSV_PATH
)


# ============================================================
# 9. CONVERT EACH CSV ROW TO TEXT
# ============================================================

def row_to_text(row):

    parts = []

    for column in df.columns:

        value = row[column]

        if pd.notna(value):

            parts.append(
                f"{column}: {value}"
            )

    return " | ".join(parts)


texts = df.apply(
    row_to_text,
    axis=1
).tolist()


# ============================================================
# 10. CREATE TRAINING CORPUS
# ============================================================

corpus = "\n<ROW>\n".join(
    texts
)


print("\n====================================")
print("CORPUS INFORMATION")
print("====================================")

print(
    "Corpus characters:",
    len(corpus)
)

print(
    "Number of records:",
    len(texts)
)


print("\nExample training text:\n")

print(
    corpus[:1000]
)


# ============================================================
# 11. CHARACTER TOKENIZER
# ============================================================

characters = sorted(
    list(
        set(corpus)
    )
)

vocab_size = len(
    characters
)


stoi = {
    ch: i
    for i, ch
    in enumerate(characters)
}


itos = {
    i: ch
    for ch, i
    in stoi.items()
}


def encode(text):

    return [
        stoi[ch]
        for ch in text
        if ch in stoi
    ]


def decode(token_ids):

    return "".join(
        itos[int(i)]
        for i in token_ids
    )


# ============================================================
# 12. ENCODE CORPUS
# ============================================================

data = torch.tensor(
    encode(corpus),
    dtype=torch.long
)


print(
    "\nVocabulary size:",
    vocab_size
)

print(
    "Total tokens:",
    len(data)
)


# ============================================================
# 13. TRAIN / VALIDATION SPLIT
# ============================================================

split_index = int(
    0.90 * len(data)
)


train_data = data[
    :split_index
]

val_data = data[
    split_index:
]


print(
    "Training tokens:",
    len(train_data)
)

print(
    "Validation tokens:",
    len(val_data)
)


# ============================================================
# 14. CHECK CORPUS SIZE
# ============================================================

if len(train_data) <= block_size + 1:

    raise ValueError(
        f"Training corpus too small for "
        f"block_size={block_size}. "
        f"Training tokens={len(train_data)}"
    )


if len(val_data) <= block_size + 1:

    print(
        "\nWARNING:"
        " Validation data is smaller than "
        "block_size."
    )

    print(
        "Training data will be used "
        "for validation batches."
    )


# ============================================================
# 15. GET TRAINING BATCH
# ============================================================

def get_batch(split):

    if split == "train":

        source = train_data

    else:

        if len(val_data) > block_size + 1:

            source = val_data

        else:

            source = train_data


    max_start = (
        len(source)
        - block_size
        - 1
    )


    indices = torch.randint(
        0,
        max_start,
        (batch_size,)
    )


    x = torch.stack([
        source[
            i:i + block_size
        ]
        for i in indices
    ])


    y = torch.stack([
        source[
            i + 1:
            i + block_size + 1
        ]
        for i in indices
    ])


    return (
        x.to(device),
        y.to(device)
    )


# ============================================================
# 16. TRANSFORMER BLOCK
# ============================================================

class TransformerBlock(
    nn.Module
):

    def __init__(
        self,
        embedding_dim,
        num_heads,
        dropout
    ):

        super().__init__()


        self.ln1 = nn.LayerNorm(
            embedding_dim
        )

        self.ln2 = nn.LayerNorm(
            embedding_dim
        )


        self.attention = (
            nn.MultiheadAttention(
                embed_dim=
                    embedding_dim,

                num_heads=
                    num_heads,

                dropout=
                    dropout,

                batch_first=True
            )
        )


        self.feed_forward = (
            nn.Sequential(

                nn.Linear(
                    embedding_dim,
                    4 * embedding_dim
                ),

                nn.GELU(),

                nn.Linear(
                    4 * embedding_dim,
                    embedding_dim
                ),

                nn.Dropout(
                    dropout
                )
            )
        )


    def forward(
        self,
        x
    ):

        T = x.size(1)


        causal_mask = (
            torch.triu(

                torch.ones(
                    T,
                    T,
                    device=x.device,
                    dtype=torch.bool
                ),

                diagonal=1
            )
        )


        normalized = (
            self.ln1(x)
        )


        attention_output, _ = (
            self.attention(

                normalized,
                normalized,
                normalized,

                attn_mask=
                    causal_mask,

                need_weights=False
            )
        )


        x = (
            x
            + attention_output
        )


        x = (
            x
            + self.feed_forward(
                self.ln2(x)
            )
        )


        return x


# ============================================================
# 17. LANGUAGE MODEL
# ============================================================

class TinyCSVLLM(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        num_heads,
        num_layers,
        block_size,
        dropout
    ):

        super().__init__()


        self.vocab_size = (
            vocab_size
        )

        self.embedding_dim = (
            embedding_dim
        )

        self.block_size = (
            block_size
        )


        self.token_embedding = (
            nn.Embedding(
                vocab_size,
                embedding_dim
            )
        )


        self.position_embedding = (
            nn.Embedding(
                block_size,
                embedding_dim
            )
        )


        self.blocks = (
            nn.ModuleList([
                TransformerBlock(
                    embedding_dim,
                    num_heads,
                    dropout
                )

                for _ in range(
                    num_layers
                )
            ])
        )


        self.final_norm = (
            nn.LayerNorm(
                embedding_dim
            )
        )


        self.lm_head = (
            nn.Linear(
                embedding_dim,
                vocab_size
            )
        )


    def forward(
        self,
        idx,
        targets=None
    ):

        B, T = idx.shape


        if T > self.block_size:

            raise ValueError(
                f"Sequence length "
                f"{T} exceeds "
                f"block_size "
                f"{self.block_size}"
            )


        positions = (
            torch.arange(
                T,
                device=idx.device
            )
        )


        token_embeddings = (
            self.token_embedding(
                idx
            )
        )


        position_embeddings = (
            self.position_embedding(
                positions
            )
        )


        x = (
            token_embeddings
            + position_embeddings
        )


        for block in self.blocks:

            x = block(x)


        x = (
            self.final_norm(x)
        )


        logits = (
            self.lm_head(x)
        )


        loss = None


        if targets is not None:

            B, T, C = (
                logits.shape
            )


            logits_flat = (
                logits.reshape(
                    B * T,
                    C
                )
            )


            targets_flat = (
                targets.reshape(
                    B * T
                )
            )


            loss = (
                F.cross_entropy(
                    logits_flat,
                    targets_flat
                )
            )


        return logits, loss


    # ========================================================
    # TEXT GENERATION
    # ========================================================

    @torch.no_grad()
    def generate(
        self,
        idx,
        max_new_tokens=300,
        temperature=0.8,
        top_k=20
    ):

        self.eval()


        for _ in range(
            max_new_tokens
        ):


            idx_context = (
                idx[
                    :,
                    -self.block_size:
                ]
            )


            logits, _ = self(
                idx_context
            )


            logits = (
                logits[
                    :,
                    -1,
                    :
                ]
            )


            logits = (
                logits
                / temperature
            )


            if top_k is not None:

                k = min(
                    top_k,
                    logits.size(-1)
                )


                values, _ = (
                    torch.topk(
                        logits,
                        k
                    )
                )


                cutoff = (
                    values[:, [-1]]
                )


                logits = (
                    torch.where(

                        logits
                        < cutoff,

                        torch.full_like(
                            logits,
                            float("-inf")
                        ),

                        logits
                    )
                )


            probabilities = (
                F.softmax(
                    logits,
                    dim=-1
                )
            )


            next_token = (
                torch.multinomial(
                    probabilities,
                    num_samples=1
                )
            )


            idx = torch.cat(
                [
                    idx,
                    next_token
                ],
                dim=1
            )


        return idx


# ============================================================
# 18. INITIALIZE MODEL
# ============================================================

model = TinyCSVLLM(

    vocab_size=
        vocab_size,

    embedding_dim=
        embedding_dim,

    num_heads=
        num_heads,

    num_layers=
        num_layers,

    block_size=
        block_size,

    dropout=
        dropout

).to(device)


# ============================================================
# 19. COUNT PARAMETERS
# ============================================================

total_parameters = sum(
    p.numel()
    for p
    in model.parameters()
)


trainable_parameters = sum(

    p.numel()

    for p
    in model.parameters()

    if p.requires_grad
)


print("\n====================================")
print("MODEL INFORMATION")
print("====================================")


print(
    f"Total parameters: "
    f"{total_parameters:,}"
)


print(
    f"Model size: "
    f"{total_parameters / 1_000_000:.3f} M"
)


print(
    f"Trainable parameters: "
    f"{trainable_parameters:,}"
)


print(
    "Transformer layers:",
    num_layers
)


print(
    "Embedding dimension:",
    embedding_dim
)


print(
    "Attention heads:",
    num_heads
)


print(
    "FFN dimension:",
    embedding_dim * 4
)


print(
    "Context length:",
    block_size
)


print(
    "Vocabulary size:",
    vocab_size
)


# ============================================================
# 20. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=
        learning_rate,

    weight_decay=
        weight_decay
)


# ============================================================
# 21. EVALUATION FUNCTION
# ============================================================

@torch.no_grad()
def estimate_loss():

    model.eval()

    results = {}


    for split in [
        "train",
        "val"
    ]:


        losses = torch.zeros(
            eval_iters
        )


        for k in range(
            eval_iters
        ):


            xb, yb = (
                get_batch(
                    split
                )
            )


            _, loss = model(
                xb,
                yb
            )


            losses[k] = (
                loss.item()
            )


        results[split] = (
            losses.mean().item()
        )


    model.train()


    return results


# ============================================================
# 22. TRAIN MODEL
# ============================================================

print("\n====================================")
print("TRAINING")
print("====================================")


model.train()


best_val_loss = (
    float("inf")
)


for step in range(
    training_steps + 1
):


    # ========================================================
    # EVALUATION
    # ========================================================

    if (
        step
        % eval_interval
        == 0

        or

        step
        == training_steps
    ):


        losses = (
            estimate_loss()
        )


        train_loss = (
            losses["train"]
        )


        val_loss = (
            losses["val"]
        )


        perplexity = (
            math.exp(
                min(
                    val_loss,
                    20
                )
            )
        )


        print(

            f"Step {step:5d}"

            f" | Train Loss: "
            f"{train_loss:.4f}"

            f" | Val Loss: "
            f"{val_loss:.4f}"

            f" | Perplexity: "
            f"{perplexity:.2f}"
        )


        # Save best weights
        if (
            val_loss
            < best_val_loss
        ):


            best_val_loss = (
                val_loss
            )


            torch.save(
                model.state_dict(),
                WEIGHTS_PATH
            )


    if (
        step
        == training_steps
    ):

        break


    # ========================================================
    # TRAINING STEP
    # ========================================================

    xb, yb = (
        get_batch(
            "train"
        )
    )


    _, loss = model(
        xb,
        yb
    )


    optimizer.zero_grad(
        set_to_none=True
    )


    loss.backward()


    torch.nn.utils.clip_grad_norm_(

        model.parameters(),

        max_norm=1.0
    )


    optimizer.step()


# ============================================================
# 23. SAVE FULL CHECKPOINT
# ============================================================

checkpoint = {

    "model_state_dict":
        model.state_dict(),

    "optimizer_state_dict":
        optimizer.state_dict(),

    "stoi":
        stoi,

    "itos":
        itos,

    "vocab_size":
        vocab_size,

    "embedding_dim":
        embedding_dim,

    "num_heads":
        num_heads,

    "num_layers":
        num_layers,

    "block_size":
        block_size,

    "dropout":
        dropout,

    "total_parameters":
        total_parameters,

    "training_steps":
        training_steps,

    "learning_rate":
        learning_rate,

    "best_val_loss":
        best_val_loss,

    "csv_columns":
        list(df.columns),

    "csv_files":
        [
            file.name
            for file in csv_files
        ]
}


torch.save(
    checkpoint,
    CHECKPOINT_PATH
)


# ============================================================
# 24. SAVE MODEL WEIGHTS
# ============================================================

torch.save(
    model.state_dict(),
    WEIGHTS_PATH
)


print("\n====================================")
print("MODEL SAVED")
print("====================================")


print(
    "Checkpoint:",
    CHECKPOINT_PATH
)


print(
    "Weights:",
    WEIGHTS_PATH
)


# ============================================================
# 25. GENERATE TEXT
# ============================================================

model.eval()


prompt = "model_type:"


prompt_ids = encode(
    prompt
)


if len(prompt_ids) == 0:

    raise ValueError(
        "Prompt contains no known "
        "characters."
    )


input_tensor = (
    torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=device
    )
)


generated_tokens = (
    model.generate(

        input_tensor,

        max_new_tokens=300,

        temperature=0.8,

        top_k=20
    )
)


generated_text = decode(
    generated_tokens[
        0
    ].tolist()
)


print("\n====================================")
print("GENERATED TEXT")
print("====================================")


print(
    generated_text
)


# ============================================================
# 26. RELOAD SAVED MODEL
# ============================================================

print("\n====================================")
print("RELOADING MODEL")
print("====================================")


loaded_checkpoint = torch.load(

    CHECKPOINT_PATH,

    map_location=device
)


loaded_stoi = (
    loaded_checkpoint[
        "stoi"
    ]
)


loaded_itos = (
    loaded_checkpoint[
        "itos"
    ]
)


loaded_model = TinyCSVLLM(

    vocab_size=
        loaded_checkpoint[
            "vocab_size"
        ],

    embedding_dim=
        loaded_checkpoint[
            "embedding_dim"
        ],

    num_heads=
        loaded_checkpoint[
            "num_heads"
        ],

    num_layers=
        loaded_checkpoint[
            "num_layers"
        ],

    block_size=
        loaded_checkpoint[
            "block_size"
        ],

    dropout=
        loaded_checkpoint[
            "dropout"
        ]

).to(device)


loaded_model.load_state_dict(

    loaded_checkpoint[
        "model_state_dict"
    ]
)


loaded_model.eval()


print(
    "Loaded parameters:",
    f"{loaded_checkpoint['total_parameters']:,}"
)


print(
    "Model successfully loaded."
)


# ============================================================
# 27. TOKENIZER FOR LOADED MODEL
# ============================================================

def loaded_encode(text):

    return [
        loaded_stoi[ch]
        for ch in text
        if ch in loaded_stoi
    ]


def loaded_decode(ids):

    return "".join(
        loaded_itos[int(i)]
        for i in ids
    )


# ============================================================
# 28. GENERATION USING RELOADED MODEL
# ============================================================

prompt = "model_type:"


prompt_ids = loaded_encode(
    prompt
)


x = torch.tensor(

    [prompt_ids],

    dtype=torch.long,

    device=device
)


generated = (
    loaded_model.generate(

        x,

        max_new_tokens=300,

        temperature=0.8,

        top_k=20
    )
)


result = loaded_decode(
    generated[
        0
    ].tolist()
)


print("\n====================================")
print("OUTPUT FROM RELOADED MODEL")
print("====================================")


print(
    result
)

Device: cuda

CSV FILES FOUND
mnist_33673b86.csv
mnist_886bbdfd.csv
mnist_9ecac72c.csv
mnist_9f56743a.csv

Total CSV files: 4

COLUMN CHECK
Reference file: mnist_33673b86.csv
Reference columns: 74

mnist_33673b86.csv: 74 columns
  Columns match

mnist_886bbdfd.csv: 74 columns
  Columns match

mnist_9ecac72c.csv: 74 columns
  Columns match

mnist_9f56743a.csv: 74 columns
  Columns match

All CSV files have compatible columns.

READING CSV FILES
mnist_33673b86.csv: 50 rows, 74 columns
mnist_886bbdfd.csv: 507 rows, 74 columns
mnist_9ecac72c.csv: 62 rows, 74 columns
mnist_9f56743a.csv: 60 rows, 74 columns

MERGED DATASET
Total CSV files: 4
Total rows: 679
Total columns: 74

Column names:
 - timestamp
 - unique_device_id
 - device_short_id
 - pc_name
 - collection_mode
 - sample_index
 - true_label
 - prediction
 - correct
 - model_type
 - parameters
 - model_flops
 - confidence_score
 - logit_margin
 - entropy
 - execution_time_sec
 - cpu_energy_kwh
 - gpu_energy_kwh
 - ram_energy_kwh
 - t

KeyboardInterrupt: 

# Load the model

In [2]:
import torch

CHECKPOINT_PATH = "csv_llm_1M_checkpoint.pth"

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load checkpoint
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device
)

# Load tokenizer
stoi = checkpoint["stoi"]
itos = checkpoint["itos"]

# Rebuild model
model = TinyCSVLLM(
    vocab_size=checkpoint["vocab_size"],
    embedding_dim=checkpoint["embedding_dim"],
    num_heads=checkpoint["num_heads"],
    num_layers=checkpoint["num_layers"],
    block_size=checkpoint["block_size"],
    dropout=checkpoint["dropout"]
).to(device)

# Load trained weights
model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Model loaded successfully.")
print("Parameters:", checkpoint["total_parameters"])

Model loaded successfully.
Parameters: 1026761


In [5]:
# ============================================================
# RUN PROMPT
# ============================================================

def encode(text):
    return [
        stoi[ch]
        for ch in text
        if ch in stoi
    ]

def decode(ids):
    return "".join(
        itos[int(i)]
        for i in ids
    )


prompt = """
model_type: qwen |
parameters: 2000000000 |
model_flops: 3000000000 |
cpu_model: Raspberry Pi 4 |
ram_total_gb: 4 |
task: predict approximate execution time |
execution_time_sec:
"""


# Convert prompt to token IDs
prompt_ids = encode(prompt)

x = torch.tensor(
    [prompt_ids],
    dtype=torch.long,
    device=device
)


# Generate output
with torch.no_grad():
    generated = model.generate(
        x,
        max_new_tokens=100,
        temperature=0.6,
        top_k=10
    )


# Decode generated tokens
result = decode(
    generated[0].tolist()
)

print("\n==============================")
print("MODEL OUTPUT")
print("==============================")

print(result)


MODEL OUTPUT

model_type: qwen |
parameters: 2000000000 |
model_flops: 3000000000 |
cpu_model: Raspberry Pi 4 |
ram_total_gb: 4 |
task: predict approximate execution time |
execution_time_sec:
re: 1.00006 | cpu_energy_kwh: 1.293331838683119e-07 | gpu_energy_kwh: 1.77683772206e-08 | ram_energy


In [8]:
prompt = """
model_type: qwen |
parameters: 2000000000 |
model_flops: 3000000000 |
cpu_model: Raspberry Pi 4 |
ram_total_gb: 4 |
execution_time_sec:
"""

result = decode(
    generated[0].tolist()
)

# Get only text after execution_time_sec:
prediction = result.split("execution_time_sec:")[-1].strip()

# Take only the first generated value
prediction = prediction.split("|")[0].strip()
prediction = prediction.split("\n")[0].strip()

print('Execution time (sec): {} s'.format(prediction))

Execution time (sec): re: 1.00006 s


# Claude code

In [87]:
"""
Character-level transformer trained on CSV telemetry rows.

Corrected version. Each substantive change is marked with a "FIX:" comment
at the point where it applies.
"""

import json
import math
import random
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# 1. CONFIGURATION
# ============================================================

CSV_FOLDER = Path(".")

# FIX: outputs go to their own directory so the merged CSV is never
# re-globbed as training input on the next run.
OUTPUT_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(exist_ok=True)

CHECKPOINT_PATH = OUTPUT_DIR / "csv_llm_1M_checkpoint.pth"
WEIGHTS_PATH = OUTPUT_DIR / "csv_llm_1M_weights.pth"
MERGED_CSV_PATH = OUTPUT_DIR / "merged_training_data.csv"

SEED = 42

# Training settings
batch_size = 16
block_size = 128
training_steps = 2000
eval_interval = 100
eval_iters = 20

learning_rate = 3e-4
min_learning_rate = 3e-5
warmup_steps = 100
weight_decay = 0.01
grad_clip = 1.0
dropout = 0.1

# Approximately 1M parameter model
embedding_dim = 128
num_heads = 4
num_layers = 5

# Data handling
FLOAT_PRECISION = 6      # significant digits kept when serializing floats
DROP_CONSTANT_COLUMNS = True
DROP_EMPTY_COLUMNS = True
DEDUPLICATE_ROWS = True
VAL_FRACTION = 0.10

ROW_START = "<ROW>\n"    # every record begins with this marker


# ============================================================
# 2. DEVICE AND SEED
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 3. FIND CSV FILES
# ============================================================

# FIX: resolve() both sides so the merged output is excluded even if
# OUTPUT_DIR sits inside CSV_FOLDER.
merged_resolved = MERGED_CSV_PATH.resolve()

csv_files = sorted(
    f for f in CSV_FOLDER.glob("*.csv")
    if f.resolve() != merged_resolved
)

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in folder: {CSV_FOLDER}")

print("\n==== CSV FILES FOUND ====")
for file in csv_files:
    print(" -", file.name)
print("Total CSV files:", len(csv_files))


# ============================================================
# 4. COLUMN CONSISTENCY CHECK
# ============================================================

reference_file = csv_files[0]
reference_columns = list(pd.read_csv(reference_file, nrows=0).columns)
reference_column_set = set(reference_columns)

print("\n==== COLUMN CHECK ====")
print(f"Reference file: {reference_file.name} ({len(reference_columns)} columns)")

all_columns_match = True

for file in csv_files:
    columns = set(pd.read_csv(file, nrows=0).columns)
    missing = reference_column_set - columns
    extra = columns - reference_column_set

    if not missing and not extra:
        print(f"  {file.name}: OK")
    else:
        all_columns_match = False
        print(f"  {file.name}: MISMATCH")
        if missing:
            print("    missing:", sorted(missing))
        if extra:
            print("    extra:  ", sorted(extra))

if not all_columns_match:
    raise ValueError("CSV files have different columns. Fix the mismatch first.")


# ============================================================
# 5. READ AND MERGE
# ============================================================

print("\n==== READING ====")

dataframes = []
for file in csv_files:
    temp_df = pd.read_csv(file)[reference_columns]
    print(f"  {file.name}: {len(temp_df)} rows")
    dataframes.append(temp_df)

df = pd.concat(dataframes, ignore_index=True)
print(f"Merged: {len(df)} rows x {len(df.columns)} columns")


# ============================================================
# 6. PRUNE USELESS COLUMNS
# ============================================================
# FIX: columns that never vary carry zero information but consume the
# large majority of every training sequence. With block_size=128, a row
# padded out with constant fields means the model almost never sees the
# fields that actually change.

original_columns = list(df.columns)

if DROP_EMPTY_COLUMNS:
    empty_cols = [c for c in df.columns if df[c].isna().all()]
    if empty_cols:
        df = df.drop(columns=empty_cols)
        print(f"\nDropped {len(empty_cols)} all-empty columns")

if DROP_CONSTANT_COLUMNS:
    const_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    if const_cols:
        df = df.drop(columns=const_cols)
        print(f"Dropped {len(const_cols)} constant columns")

if df.shape[1] == 0:
    raise ValueError(
        "Every column was constant or empty. There is nothing to learn from "
        "this dataset. Collect data across varied configurations first."
    )

print(f"Retained {df.shape[1]} of {len(original_columns)} columns:")
print(" ", ", ".join(df.columns))

if DEDUPLICATE_ROWS:
    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    if len(df) < before:
        print(f"Dropped {before - len(df)} exact duplicate rows")

df.to_csv(MERGED_CSV_PATH, index=False)
print("Merged CSV saved to:", MERGED_CSV_PATH)


# ============================================================
# 7. SERIALIZE ROWS TO TEXT
# ============================================================

def format_value(value):
    """FIX: cap float precision.

    Pandas renders 0.01818181818181818 in full, so a single field can eat
    20+ characters of a 128-character context window. Six significant
    digits preserves the information at a fraction of the length.
    """
    if isinstance(value, float):
        if value.is_integer() and abs(value) < 1e15:
            return str(int(value))
        return f"{value:.{FLOAT_PRECISION}g}"
    return str(value)


columns = list(df.columns)


def row_to_text(row):
    parts = [
        f"{col}: {format_value(row[col])}"
        for col in columns
        if pd.notna(row[col])
    ]
    return " | ".join(parts)


texts = df.apply(row_to_text, axis=1).tolist()


# ============================================================
# 8. ROW-LEVEL TRAIN / VALIDATION SPLIT
# ============================================================
# FIX: the original split the *character stream* at 90%, which cuts a row
# in half and puts its first part in train and its tail in val. Splitting
# whole rows is the minimum correct thing to do.

indices = list(range(len(texts)))
random.Random(SEED).shuffle(indices)

n_val = max(1, int(VAL_FRACTION * len(texts)))
val_indices = set(indices[:n_val])

train_texts = [t for i, t in enumerate(texts) if i not in val_indices]
val_texts = [t for i, t in enumerate(texts) if i in val_indices]


def build_corpus(rows):
    return "".join(ROW_START + r + "\n" for r in rows)


train_corpus = build_corpus(train_texts)
val_corpus = build_corpus(val_texts)
full_corpus = train_corpus + val_corpus

print("\n==== CORPUS ====")
print(f"Rows: {len(texts)} (train {len(train_texts)}, val {len(val_texts)})")
print(f"Characters: {len(full_corpus)} (train {len(train_corpus)}, val {len(val_corpus)})")
print("\nExample record:\n")
print(train_corpus[:600])


# ============================================================
# 9. CHARACTER TOKENIZER
# ============================================================
# Vocabulary is built from the full corpus so validation characters are
# never out-of-vocabulary.

characters = sorted(set(full_corpus))
vocab_size = len(characters)

stoi = {ch: i for i, ch in enumerate(characters)}
itos = {i: ch for ch, i in stoi.items()}


def encode(text):
    return [stoi[ch] for ch in text if ch in stoi]


def decode(token_ids):
    return "".join(itos[int(i)] for i in token_ids)


train_data = torch.tensor(encode(train_corpus), dtype=torch.long)
val_data = torch.tensor(encode(val_corpus), dtype=torch.long)

print(f"\nVocabulary size: {vocab_size}")
print(f"Train tokens: {len(train_data)}, Val tokens: {len(val_data)}")


# ============================================================
# 10. SIZE SANITY CHECKS
# ============================================================

if len(train_data) <= block_size + 1:
    raise ValueError(
        f"Training corpus too small for block_size={block_size} "
        f"(tokens={len(train_data)}). Reduce block_size or add data."
    )

have_real_val = len(val_data) > block_size + 1

if not have_real_val:
    print(
        "\nWARNING: validation corpus is smaller than block_size. "
        "Validation loss will be computed on training data and is "
        "NOT a measure of generalization."
    )

# FIX: this is the check the original script most needed. A 1M-parameter
# model against a corpus this small memorizes rather than generalizes.
tokens_per_param = len(train_data) / 1_020_000
if tokens_per_param < 1.0:
    print(
        f"\nWARNING: ~{len(train_data):,} training tokens for a ~1.0M "
        f"parameter model ({tokens_per_param:.3f} tokens/parameter).\n"
        "         The model has far more capacity than data. Expect it to\n"
        "         memorize the rows verbatim. Treat generated output as\n"
        "         recall, not synthesis."
    )


# ============================================================
# 11. BATCHING
# ============================================================

def get_batch(split):
    source = train_data if (split == "train" or not have_real_val) else val_data

    # FIX: original used randint(0, max_start), excluding the final valid
    # window. high is exclusive, so pass max_start + 1.
    max_start = len(source) - block_size - 1

    starts = torch.randint(0, max_start + 1, (batch_size,))

    x = torch.stack([source[i:i + block_size] for i in starts])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in starts])

    return x.to(device), y.to(device)


# ============================================================
# 12. TRANSFORMER BLOCK
# ============================================================

class TransformerBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads, dropout):
        super().__init__()

        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)

        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

        # FIX: the original applied dropout inside the feed-forward branch
        # only. The attention branch had no residual dropout at all.
        self.attn_dropout = nn.Dropout(dropout)

        self.feed_forward = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.GELU(),
            nn.Linear(4 * embedding_dim, embedding_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, causal_mask):
        normalized = self.ln1(x)

        attn_out, _ = self.attention(
            normalized, normalized, normalized,
            attn_mask=causal_mask,
            need_weights=False,
        )

        x = x + self.attn_dropout(attn_out)
        x = x + self.feed_forward(self.ln2(x))

        return x


# ============================================================
# 13. LANGUAGE MODEL
# ============================================================

class TinyCSVLLM(nn.Module):

    def __init__(self, vocab_size, embedding_dim, num_heads,
                 num_layers, block_size, dropout):
        super().__init__()

        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.block_size = block_size

        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(block_size, embedding_dim)
        self.embedding_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(embedding_dim, num_heads, dropout)
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(embedding_dim)
        self.lm_head = nn.Linear(embedding_dim, vocab_size)

        # FIX: the causal mask was rebuilt inside every block on every
        # forward pass (num_layers allocations per step). Cache it once.
        mask = torch.triu(
            torch.ones(block_size, block_size, dtype=torch.bool),
            diagonal=1,
        )
        self.register_buffer("causal_mask", mask, persistent=False)

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        if T > self.block_size:
            raise ValueError(
                f"Sequence length {T} exceeds block_size {self.block_size}"
            )

        positions = torch.arange(T, device=idx.device)

        x = self.token_embedding(idx) + self.position_embedding(positions)
        x = self.embedding_dropout(x)

        mask = self.causal_mask[:T, :T]

        for block in self.blocks:
            x = block(x, mask)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(B * T, self.vocab_size),
                targets.reshape(B * T),
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=300, temperature=0.8, top_k=20):
        # FIX: original called self.eval() and never restored the previous
        # mode, silently disabling dropout for any training that followed.
        was_training = self.training
        self.eval()

        try:
            for _ in range(max_new_tokens):
                idx_context = idx[:, -self.block_size:]

                logits, _ = self(idx_context)
                logits = logits[:, -1, :] / max(temperature, 1e-6)

                if top_k is not None:
                    k = min(top_k, logits.size(-1))
                    values, _ = torch.topk(logits, k)
                    cutoff = values[:, [-1]]
                    logits = logits.masked_fill(logits < cutoff, float("-inf"))

                probabilities = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probabilities, num_samples=1)

                idx = torch.cat([idx, next_token], dim=1)
        finally:
            if was_training:
                self.train()

        return idx


# ============================================================
# 14. INITIALIZE
# ============================================================

model = TinyCSVLLM(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    block_size=block_size,
    dropout=dropout,
).to(device)

total_parameters = sum(p.numel() for p in model.parameters())

print("\n==== MODEL ====")
print(f"Parameters: {total_parameters:,} ({total_parameters / 1e6:.3f} M)")
print(f"Layers: {num_layers} | Heads: {num_heads} | Dim: {embedding_dim}")
print(f"Context: {block_size} | Vocab: {vocab_size}")


# ============================================================
# 15. OPTIMIZER
# ============================================================
# FIX: weight decay should not be applied to LayerNorm parameters, biases,
# or embeddings. The original decayed every parameter uniformly.

decay_params, no_decay_params = [], []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if param.dim() >= 2 and "embedding" not in name:
        decay_params.append(param)
    else:
        no_decay_params.append(param)

optimizer = torch.optim.AdamW(
    [
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=learning_rate,
    betas=(0.9, 0.95),
)


def lr_at(step):
    """Linear warmup then cosine decay."""
    if step < warmup_steps:
        return learning_rate * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, training_steps - warmup_steps)
    progress = min(1.0, progress)
    coeff = 0.5 * (1.0 + math.cos(math.pi * progress))
    return min_learning_rate + coeff * (learning_rate - min_learning_rate)


# ============================================================
# 16. EVALUATION
# ============================================================

@torch.no_grad()
def estimate_loss():
    model.eval()
    results = {}

    for split in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        results[split] = losses.mean().item()

    model.train()
    return results


def save_checkpoint(path, step, val_loss):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "stoi": stoi,
            "itos": itos,
            "vocab_size": vocab_size,
            "embedding_dim": embedding_dim,
            "num_heads": num_heads,
            "num_layers": num_layers,
            "block_size": block_size,
            "dropout": dropout,
            "total_parameters": total_parameters,
            "step": step,
            "best_val_loss": val_loss,
            "learning_rate": learning_rate,
            "csv_columns": list(df.columns),
            "dropped_columns": [c for c in original_columns if c not in df.columns],
            "csv_files": [f.name for f in csv_files],
            "row_start": ROW_START,
            "float_precision": FLOAT_PRECISION,
        },
        path,
    )


# ============================================================
# 17. TRAIN
# ============================================================

print("\n==== TRAINING ====")

model.train()
best_val_loss = float("inf")
best_step = -1

for step in range(training_steps + 1):

    if step % eval_interval == 0 or step == training_steps:
        losses = estimate_loss()
        perplexity = math.exp(min(losses["val"], 20))

        marker = ""
        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            best_step = step
            # FIX: save the FULL checkpoint at the best step, not just the
            # bare state_dict. The original saved best weights during
            # training and then unconditionally overwrote that file with
            # the final (worse) weights afterwards.
            save_checkpoint(CHECKPOINT_PATH, step, best_val_loss)
            torch.save(model.state_dict(), WEIGHTS_PATH)
            marker = "  <- best"

        print(
            f"Step {step:5d} | Train {losses['train']:.4f} "
            f"| Val {losses['val']:.4f} | PPL {perplexity:7.2f}{marker}"
        )

    if step == training_steps:
        break

    for group in optimizer.param_groups:
        group["lr"] = lr_at(step)

    xb, yb = get_batch("train")
    _, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
    optimizer.step()

print(f"\nBest val loss {best_val_loss:.4f} at step {best_step}")
print("Checkpoint:", CHECKPOINT_PATH)
print("Weights:   ", WEIGHTS_PATH)


# ============================================================
# 18. RELOAD BEST CHECKPOINT
# ============================================================
# FIX: the original generated text from the final model but described it
# as the saved one. Load the best checkpoint explicitly before generating.

# weights_only=False is required because the checkpoint stores the
# tokenizer dictionaries alongside the tensors.
loaded_checkpoint = torch.load(
    CHECKPOINT_PATH, map_location=device, weights_only=False
)

loaded_stoi = loaded_checkpoint["stoi"]
loaded_itos = loaded_checkpoint["itos"]

loaded_model = TinyCSVLLM(
    vocab_size=loaded_checkpoint["vocab_size"],
    embedding_dim=loaded_checkpoint["embedding_dim"],
    num_heads=loaded_checkpoint["num_heads"],
    num_layers=loaded_checkpoint["num_layers"],
    block_size=loaded_checkpoint["block_size"],
    dropout=loaded_checkpoint["dropout"],
).to(device)

loaded_model.load_state_dict(loaded_checkpoint["model_state_dict"])
loaded_model.eval()

print(f"\nReloaded model from step {loaded_checkpoint['step']} "
      f"({loaded_checkpoint['total_parameters']:,} parameters)")


# ============================================================
# 19. GENERATE
# ============================================================
# FIX: the original prompted with "model_type:" starting at position 0.
# That string never appears at the start of a record, so the model was
# asked to continue from a context it never saw during training. Prompt
# with the row-start marker instead.

prompt = ROW_START

prompt_ids = [loaded_stoi[ch] for ch in prompt if ch in loaded_stoi]
if not prompt_ids:
    raise ValueError("Prompt contains no known characters.")

x = torch.tensor([prompt_ids], dtype=torch.long, device=device)

generated = loaded_model.generate(
    x, max_new_tokens=600, temperature=0.8, top_k=20
)

result = "".join(loaded_itos[int(i)] for i in generated[0].tolist())

print("\n==== GENERATED RECORD ====")
print(result)


# ============================================================
# 20. MEMORIZATION CHECK
# ============================================================
# The single most useful diagnostic for a model this size on data this
# small: is the output novel, or is it reciting a training row?

generated_body = result[len(ROW_START):].split(ROW_START)[0].strip()

import difflib

best_ratio, best_match = 0.0, ""
for t in train_texts:
    ratio = difflib.SequenceMatcher(None, generated_body, t).ratio()
    if ratio > best_ratio:
        best_ratio, best_match = ratio, t

print("\n==== MEMORIZATION CHECK ====")
print(f"Closest training row similarity: {best_ratio:.3f}")
if best_ratio > 0.90:
    print("The model is reproducing training rows almost verbatim.")
    print("This is expected at this data scale and is not a bug in the code.")

stats = {
    "rows": len(texts),
    "train_rows": len(train_texts),
    "val_rows": len(val_texts),
    "train_tokens": len(train_data),
    "val_tokens": len(val_data),
    "vocab_size": vocab_size,
    "parameters": total_parameters,
    "best_val_loss": best_val_loss,
    "best_step": best_step,
    "generation_similarity_to_train": best_ratio,
}

with open(OUTPUT_DIR / "run_stats.json", "w") as fh:
    json.dump(stats, fh, indent=2)

print("\nRun stats written to:", OUTPUT_DIR / "run_stats.json")


Device: cuda

==== CSV FILES FOUND ====
 - merged_training_data.csv
 - mnist_33673b86.csv
 - mnist_886bbdfd.csv
 - mnist_9ecac72c.csv
 - mnist_9f56743a.csv
Total CSV files: 5

==== COLUMN CHECK ====
Reference file: merged_training_data.csv (74 columns)
  merged_training_data.csv: OK
  mnist_33673b86.csv: OK
  mnist_886bbdfd.csv: OK
  mnist_9ecac72c.csv: OK
  mnist_9f56743a.csv: OK

==== READING ====
  merged_training_data.csv: 679 rows
  mnist_33673b86.csv: 50 rows
  mnist_886bbdfd.csv: 507 rows
  mnist_9ecac72c.csv: 62 rows
  mnist_9f56743a.csv: 60 rows
Merged: 1358 rows x 74 columns

Dropped 2 all-empty columns
Dropped 1 constant columns
Retained 71 of 74 columns:
  timestamp, unique_device_id, device_short_id, pc_name, sample_index, true_label, prediction, correct, model_type, parameters, model_flops, confidence_score, logit_margin, entropy, execution_time_sec, cpu_energy_kwh, gpu_energy_kwh, ram_energy_kwh, total_energy_kwh, total_emissions_kg, carbon_intensity_kgco2_kwh, codecarbo

In [88]:
# ============================================================
# RUN PROMPT
# ============================================================

def encode(text):
    return [
        stoi[ch]
        for ch in text
        if ch in stoi
    ]

def decode(ids):
    return "".join(
        itos[int(i)]
        for i in ids
    )


prompt = """
model_type: qwen |
parameters: 2000000000 |
model_flops: 3000000000 |
cpu_model: Raspberry Pi 4 |
ram_total_gb: 4 |
task: predict approximate execution time |
execution_time_sec:
"""


# Convert prompt to token IDs
prompt_ids = encode(prompt)

x = torch.tensor(
    [prompt_ids],
    dtype=torch.long,
    device=device
)


# Generate output
with torch.no_grad():
    generated = model.generate(
        x,
        max_new_tokens=100,
        temperature=0.6,
        top_k=10
    )


# Decode generated tokens
result = decode(
    generated[0].tolist()
)

print("\n==============================")
print("MODEL OUTPUT")
print("==============================")

print(result)


MODEL OUTPUT

model_type: qwen |
parameters: 2000000000 |
model_flops: 3000000000 |
cpu_model: Raspberry Pi 4 |
ram_total_gb: 4 |
task: predict approximate execution time |
execution_time_sec:
: 2 | cpu_endergy_pct: 0 | cpu_model: Intel(R) Core(TM) Ultra 5 245 | cpu_architecture: X86_64 | cpu


# claude

In [1]:
"""
Character-level transformer trained on CSV telemetry rows.

Corrected version. Each substantive change is marked with a "FIX:" comment
at the point where it applies.
"""

import json
import math
import random
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# 1. CONFIGURATION
# ============================================================

CSV_FOLDER = Path(".")

# FIX: outputs go to their own directory so the merged CSV is never
# re-globbed as training input on the next run.
OUTPUT_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(exist_ok=True)

CHECKPOINT_PATH = OUTPUT_DIR / "csv_llm_1M_checkpoint.pth"
WEIGHTS_PATH = OUTPUT_DIR / "csv_llm_1M_weights.pth"
MERGED_CSV_PATH = OUTPUT_DIR / "merged_training_data.csv"

SEED = 42

# Training settings
batch_size = 16
block_size = 128
training_steps = 2000
eval_interval = 100
eval_iters = 20

learning_rate = 3e-4
min_learning_rate = 3e-5
warmup_steps = 100
weight_decay = 0.01
grad_clip = 1.0
dropout = 0.1

# Approximately 1M parameter model
embedding_dim = 128
num_heads = 4
num_layers = 5

# Data handling
FLOAT_PRECISION = 6      # significant digits kept when serializing floats
DROP_CONSTANT_COLUMNS = True
DROP_EMPTY_COLUMNS = True
DEDUPLICATE_ROWS = True
VAL_FRACTION = 0.10

ROW_START = "<ROW>\n"    # every record begins with this marker


# ============================================================
# 2. DEVICE AND SEED
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 3. FIND CSV FILES
# ============================================================

# FIX: resolve() both sides so the merged output is excluded even if
# OUTPUT_DIR sits inside CSV_FOLDER.
merged_resolved = MERGED_CSV_PATH.resolve()

csv_files = sorted(
    f for f in CSV_FOLDER.glob("*.csv")
    if f.resolve() != merged_resolved
)

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in folder: {CSV_FOLDER}")

print("\n==== CSV FILES FOUND ====")
for file in csv_files:
    print(" -", file.name)
print("Total CSV files:", len(csv_files))


# ============================================================
# 4. COLUMN CONSISTENCY CHECK
# ============================================================

reference_file = csv_files[0]
reference_columns = list(pd.read_csv(reference_file, nrows=0).columns)
reference_column_set = set(reference_columns)

print("\n==== COLUMN CHECK ====")
print(f"Reference file: {reference_file.name} ({len(reference_columns)} columns)")

all_columns_match = True

for file in csv_files:
    columns = set(pd.read_csv(file, nrows=0).columns)
    missing = reference_column_set - columns
    extra = columns - reference_column_set

    if not missing and not extra:
        print(f"  {file.name}: OK")
    else:
        all_columns_match = False
        print(f"  {file.name}: MISMATCH")
        if missing:
            print("    missing:", sorted(missing))
        if extra:
            print("    extra:  ", sorted(extra))

if not all_columns_match:
    raise ValueError("CSV files have different columns. Fix the mismatch first.")


# ============================================================
# 5. READ AND MERGE
# ============================================================

print("\n==== READING ====")

dataframes = []
for file in csv_files:
    temp_df = pd.read_csv(file)[reference_columns]
    print(f"  {file.name}: {len(temp_df)} rows")
    dataframes.append(temp_df)

df = pd.concat(dataframes, ignore_index=True)
print(f"Merged: {len(df)} rows x {len(df.columns)} columns")


# ============================================================
# 6. PRUNE USELESS COLUMNS
# ============================================================
# FIX: columns that never vary carry zero information but consume the
# large majority of every training sequence. With block_size=128, a row
# padded out with constant fields means the model almost never sees the
# fields that actually change.

original_columns = list(df.columns)

if DROP_EMPTY_COLUMNS:
    empty_cols = [c for c in df.columns if df[c].isna().all()]
    if empty_cols:
        df = df.drop(columns=empty_cols)
        print(f"\nDropped {len(empty_cols)} all-empty columns")

if DROP_CONSTANT_COLUMNS:
    const_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    if const_cols:
        df = df.drop(columns=const_cols)
        print(f"Dropped {len(const_cols)} constant columns")

if df.shape[1] == 0:
    raise ValueError(
        "Every column was constant or empty. There is nothing to learn from "
        "this dataset. Collect data across varied configurations first."
    )

print(f"Retained {df.shape[1]} of {len(original_columns)} columns:")
print(" ", ", ".join(df.columns))

if DEDUPLICATE_ROWS:
    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    if len(df) < before:
        print(f"Dropped {before - len(df)} exact duplicate rows")

df.to_csv(MERGED_CSV_PATH, index=False)
print("Merged CSV saved to:", MERGED_CSV_PATH)


# ============================================================
# 7. SERIALIZE ROWS TO TEXT
# ============================================================

def format_value(value):
    """FIX: cap float precision.

    Pandas renders 0.01818181818181818 in full, so a single field can eat
    20+ characters of a 128-character context window. Six significant
    digits preserves the information at a fraction of the length.
    """
    if isinstance(value, float):
        if value.is_integer() and abs(value) < 1e15:
            return str(int(value))
        return f"{value:.{FLOAT_PRECISION}g}"
    return str(value)


columns = list(df.columns)


def row_to_text(row):
    parts = [
        f"{col}: {format_value(row[col])}"
        for col in columns
        if pd.notna(row[col])
    ]
    return " | ".join(parts)


texts = df.apply(row_to_text, axis=1).tolist()


# ============================================================
# 8. ROW-LEVEL TRAIN / VALIDATION SPLIT
# ============================================================
# FIX: the original split the *character stream* at 90%, which cuts a row
# in half and puts its first part in train and its tail in val. Splitting
# whole rows is the minimum correct thing to do.

indices = list(range(len(texts)))
random.Random(SEED).shuffle(indices)

n_val = max(1, int(VAL_FRACTION * len(texts)))
val_indices = set(indices[:n_val])

train_texts = [t for i, t in enumerate(texts) if i not in val_indices]
val_texts = [t for i, t in enumerate(texts) if i in val_indices]


def build_corpus(rows):
    return "".join(ROW_START + r + "\n" for r in rows)


train_corpus = build_corpus(train_texts)
val_corpus = build_corpus(val_texts)
full_corpus = train_corpus + val_corpus

print("\n==== CORPUS ====")
print(f"Rows: {len(texts)} (train {len(train_texts)}, val {len(val_texts)})")
print(f"Characters: {len(full_corpus)} (train {len(train_corpus)}, val {len(val_corpus)})")
print("\nExample record:\n")
print(train_corpus[:600])


# ============================================================
# 9. CHARACTER TOKENIZER
# ============================================================
# Vocabulary is built from the full corpus so validation characters are
# never out-of-vocabulary.

characters = sorted(set(full_corpus))
vocab_size = len(characters)

stoi = {ch: i for i, ch in enumerate(characters)}
itos = {i: ch for ch, i in stoi.items()}


def encode(text):
    return [stoi[ch] for ch in text if ch in stoi]


def decode(token_ids):
    return "".join(itos[int(i)] for i in token_ids)


train_data = torch.tensor(encode(train_corpus), dtype=torch.long)
val_data = torch.tensor(encode(val_corpus), dtype=torch.long)

print(f"\nVocabulary size: {vocab_size}")
print(f"Train tokens: {len(train_data)}, Val tokens: {len(val_data)}")


# ============================================================
# 10. SIZE SANITY CHECKS
# ============================================================

if len(train_data) <= block_size + 1:
    raise ValueError(
        f"Training corpus too small for block_size={block_size} "
        f"(tokens={len(train_data)}). Reduce block_size or add data."
    )

have_real_val = len(val_data) > block_size + 1

if not have_real_val:
    print(
        "\nWARNING: validation corpus is smaller than block_size. "
        "Validation loss will be computed on training data and is "
        "NOT a measure of generalization."
    )

# FIX: this is the check the original script most needed. A 1M-parameter
# model against a corpus this small memorizes rather than generalizes.
tokens_per_param = len(train_data) / 1_020_000
if tokens_per_param < 1.0:
    print(
        f"\nWARNING: ~{len(train_data):,} training tokens for a ~1.0M "
        f"parameter model ({tokens_per_param:.3f} tokens/parameter).\n"
        "         The model has far more capacity than data. Expect it to\n"
        "         memorize the rows verbatim. Treat generated output as\n"
        "         recall, not synthesis."
    )


# ============================================================
# 11. BATCHING
# ============================================================

def get_batch(split):
    source = train_data if (split == "train" or not have_real_val) else val_data

    # FIX: original used randint(0, max_start), excluding the final valid
    # window. high is exclusive, so pass max_start + 1.
    max_start = len(source) - block_size - 1

    starts = torch.randint(0, max_start + 1, (batch_size,))

    x = torch.stack([source[i:i + block_size] for i in starts])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in starts])

    return x.to(device), y.to(device)


# ============================================================
# 12. TRANSFORMER BLOCK
# ============================================================

class TransformerBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads, dropout):
        super().__init__()

        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)

        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

        # FIX: the original applied dropout inside the feed-forward branch
        # only. The attention branch had no residual dropout at all.
        self.attn_dropout = nn.Dropout(dropout)

        self.feed_forward = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.GELU(),
            nn.Linear(4 * embedding_dim, embedding_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, causal_mask):
        normalized = self.ln1(x)

        attn_out, _ = self.attention(
            normalized, normalized, normalized,
            attn_mask=causal_mask,
            need_weights=False,
        )

        x = x + self.attn_dropout(attn_out)
        x = x + self.feed_forward(self.ln2(x))

        return x


# ============================================================
# 13. LANGUAGE MODEL
# ============================================================

class TinyCSVLLM(nn.Module):

    def __init__(self, vocab_size, embedding_dim, num_heads,
                 num_layers, block_size, dropout):
        super().__init__()

        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.block_size = block_size

        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(block_size, embedding_dim)
        self.embedding_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(embedding_dim, num_heads, dropout)
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(embedding_dim)
        self.lm_head = nn.Linear(embedding_dim, vocab_size)

        # FIX: the causal mask was rebuilt inside every block on every
        # forward pass (num_layers allocations per step). Cache it once.
        mask = torch.triu(
            torch.ones(block_size, block_size, dtype=torch.bool),
            diagonal=1,
        )
        self.register_buffer("causal_mask", mask, persistent=False)

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        if T > self.block_size:
            raise ValueError(
                f"Sequence length {T} exceeds block_size {self.block_size}"
            )

        positions = torch.arange(T, device=idx.device)

        x = self.token_embedding(idx) + self.position_embedding(positions)
        x = self.embedding_dropout(x)

        mask = self.causal_mask[:T, :T]

        for block in self.blocks:
            x = block(x, mask)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(B * T, self.vocab_size),
                targets.reshape(B * T),
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=300, temperature=0.8, top_k=20):
        # FIX: original called self.eval() and never restored the previous
        # mode, silently disabling dropout for any training that followed.
        was_training = self.training
        self.eval()

        try:
            for _ in range(max_new_tokens):
                idx_context = idx[:, -self.block_size:]

                logits, _ = self(idx_context)
                logits = logits[:, -1, :] / max(temperature, 1e-6)

                if top_k is not None:
                    k = min(top_k, logits.size(-1))
                    values, _ = torch.topk(logits, k)
                    cutoff = values[:, [-1]]
                    logits = logits.masked_fill(logits < cutoff, float("-inf"))

                probabilities = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probabilities, num_samples=1)

                idx = torch.cat([idx, next_token], dim=1)
        finally:
            if was_training:
                self.train()

        return idx


# ============================================================
# 14. INITIALIZE
# ============================================================

model = TinyCSVLLM(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    block_size=block_size,
    dropout=dropout,
).to(device)

total_parameters = sum(p.numel() for p in model.parameters())

print("\n==== MODEL ====")
print(f"Parameters: {total_parameters:,} ({total_parameters / 1e6:.3f} M)")
print(f"Layers: {num_layers} | Heads: {num_heads} | Dim: {embedding_dim}")
print(f"Context: {block_size} | Vocab: {vocab_size}")


# ============================================================
# 15. OPTIMIZER
# ============================================================
# FIX: weight decay should not be applied to LayerNorm parameters, biases,
# or embeddings. The original decayed every parameter uniformly.

decay_params, no_decay_params = [], []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if param.dim() >= 2 and "embedding" not in name:
        decay_params.append(param)
    else:
        no_decay_params.append(param)

optimizer = torch.optim.AdamW(
    [
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=learning_rate,
    betas=(0.9, 0.95),
)


def lr_at(step):
    """Linear warmup then cosine decay."""
    if step < warmup_steps:
        return learning_rate * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, training_steps - warmup_steps)
    progress = min(1.0, progress)
    coeff = 0.5 * (1.0 + math.cos(math.pi * progress))
    return min_learning_rate + coeff * (learning_rate - min_learning_rate)


# ============================================================
# 16. EVALUATION
# ============================================================

@torch.no_grad()
def estimate_loss():
    model.eval()
    results = {}

    for split in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        results[split] = losses.mean().item()

    model.train()
    return results


def save_checkpoint(path, step, val_loss):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "stoi": stoi,
            "itos": itos,
            "vocab_size": vocab_size,
            "embedding_dim": embedding_dim,
            "num_heads": num_heads,
            "num_layers": num_layers,
            "block_size": block_size,
            "dropout": dropout,
            "total_parameters": total_parameters,
            "step": step,
            "best_val_loss": val_loss,
            "learning_rate": learning_rate,
            "csv_columns": list(df.columns),
            "dropped_columns": [c for c in original_columns if c not in df.columns],
            "csv_files": [f.name for f in csv_files],
            "row_start": ROW_START,
            "float_precision": FLOAT_PRECISION,
        },
        path,
    )


# ============================================================
# 17. TRAIN
# ============================================================

print("\n==== TRAINING ====")

model.train()
best_val_loss = float("inf")
best_step = -1

for step in range(training_steps + 1):

    if step % eval_interval == 0 or step == training_steps:
        losses = estimate_loss()
        perplexity = math.exp(min(losses["val"], 20))

        marker = ""
        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            best_step = step
            # FIX: save the FULL checkpoint at the best step, not just the
            # bare state_dict. The original saved best weights during
            # training and then unconditionally overwrote that file with
            # the final (worse) weights afterwards.
            save_checkpoint(CHECKPOINT_PATH, step, best_val_loss)
            torch.save(model.state_dict(), WEIGHTS_PATH)
            marker = "  <- best"

        print(
            f"Step {step:5d} | Train {losses['train']:.4f} "
            f"| Val {losses['val']:.4f} | PPL {perplexity:7.2f}{marker}"
        )

    if step == training_steps:
        break

    for group in optimizer.param_groups:
        group["lr"] = lr_at(step)

    xb, yb = get_batch("train")
    _, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
    optimizer.step()

print(f"\nBest val loss {best_val_loss:.4f} at step {best_step}")
print("Checkpoint:", CHECKPOINT_PATH)
print("Weights:   ", WEIGHTS_PATH)


# ============================================================
# 18. RELOAD BEST CHECKPOINT
# ============================================================
# FIX: the original generated text from the final model but described it
# as the saved one. Load the best checkpoint explicitly before generating.

# weights_only=False is required because the checkpoint stores the
# tokenizer dictionaries alongside the tensors.
loaded_checkpoint = torch.load(
    CHECKPOINT_PATH, map_location=device, weights_only=False
)

loaded_stoi = loaded_checkpoint["stoi"]
loaded_itos = loaded_checkpoint["itos"]

loaded_model = TinyCSVLLM(
    vocab_size=loaded_checkpoint["vocab_size"],
    embedding_dim=loaded_checkpoint["embedding_dim"],
    num_heads=loaded_checkpoint["num_heads"],
    num_layers=loaded_checkpoint["num_layers"],
    block_size=loaded_checkpoint["block_size"],
    dropout=loaded_checkpoint["dropout"],
).to(device)

loaded_model.load_state_dict(loaded_checkpoint["model_state_dict"])
loaded_model.eval()

print(f"\nReloaded model from step {loaded_checkpoint['step']} "
      f"({loaded_checkpoint['total_parameters']:,} parameters)")


# ============================================================
# 19. GENERATE
# ============================================================
# FIX: the original prompted with "model_type:" starting at position 0.
# That string never appears at the start of a record, so the model was
# asked to continue from a context it never saw during training. Prompt
# with the row-start marker instead.

prompt = ROW_START

prompt_ids = [loaded_stoi[ch] for ch in prompt if ch in loaded_stoi]
if not prompt_ids:
    raise ValueError("Prompt contains no known characters.")

x = torch.tensor([prompt_ids], dtype=torch.long, device=device)

generated = loaded_model.generate(
    x, max_new_tokens=600, temperature=0.8, top_k=20
)

result = "".join(loaded_itos[int(i)] for i in generated[0].tolist())

print("\n==== GENERATED RECORD ====")
print(result)


# ============================================================
# 20. MEMORIZATION CHECK
# ============================================================
# The single most useful diagnostic for a model this size on data this
# small: is the output novel, or is it reciting a training row?

generated_body = result[len(ROW_START):].split(ROW_START)[0].strip()

import difflib

best_ratio, best_match = 0.0, ""
for t in train_texts:
    ratio = difflib.SequenceMatcher(None, generated_body, t).ratio()
    if ratio > best_ratio:
        best_ratio, best_match = ratio, t

print("\n==== MEMORIZATION CHECK ====")
print(f"Closest training row similarity: {best_ratio:.3f}")
if best_ratio > 0.90:
    print("The model is reproducing training rows almost verbatim.")
    print("This is expected at this data scale and is not a bug in the code.")

stats = {
    "rows": len(texts),
    "train_rows": len(train_texts),
    "val_rows": len(val_texts),
    "train_tokens": len(train_data),
    "val_tokens": len(val_data),
    "vocab_size": vocab_size,
    "parameters": total_parameters,
    "best_val_loss": best_val_loss,
    "best_step": best_step,
    "generation_similarity_to_train": best_ratio,
}

with open(OUTPUT_DIR / "run_stats.json", "w") as fh:
    json.dump(stats, fh, indent=2)

print("\nRun stats written to:", OUTPUT_DIR / "run_stats.json")


Device: cuda

==== CSV FILES FOUND ====
 - merged_training_data.csv
 - mnist_33673b86.csv
 - mnist_886bbdfd.csv
 - mnist_9ecac72c.csv
 - mnist_9f56743a.csv
Total CSV files: 5

==== COLUMN CHECK ====
Reference file: merged_training_data.csv (74 columns)
  merged_training_data.csv: OK
  mnist_33673b86.csv: OK
  mnist_886bbdfd.csv: OK
  mnist_9ecac72c.csv: OK
  mnist_9f56743a.csv: OK

==== READING ====
  merged_training_data.csv: 679 rows
  mnist_33673b86.csv: 50 rows
  mnist_886bbdfd.csv: 507 rows
  mnist_9ecac72c.csv: 62 rows
  mnist_9f56743a.csv: 60 rows
Merged: 1358 rows x 74 columns

Dropped 2 all-empty columns
Dropped 1 constant columns
Retained 71 of 74 columns:
  timestamp, unique_device_id, device_short_id, pc_name, sample_index, true_label, prediction, correct, model_type, parameters, model_flops, confidence_score, logit_margin, entropy, execution_time_sec, cpu_energy_kwh, gpu_energy_kwh, ram_energy_kwh, total_energy_kwh, total_emissions_kg, carbon_intensity_kgco2_kwh, codecarbo

In [2]:
"""
Load a trained csv_llm checkpoint and prompt it.

Standalone: does not import the training script (that script runs training
on import, since it is a flat module). The model class below must stay in
sync with the one in train_csv_llm.py — architecture hyperparameters are
read from the checkpoint, but the class structure itself is duplicated.
"""

import re
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F


CHECKPOINT_PATH = Path("artifacts/csv_llm_1M_checkpoint.pth")

device = "cuda" if torch.cuda.is_available() else "cpu"


# ============================================================
# 1. MODEL DEFINITION (must match training)
# ============================================================

class TransformerBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)
        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.attn_dropout = nn.Dropout(dropout)
        self.feed_forward = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.GELU(),
            nn.Linear(4 * embedding_dim, embedding_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, causal_mask):
        normalized = self.ln1(x)
        attn_out, _ = self.attention(
            normalized, normalized, normalized,
            attn_mask=causal_mask, need_weights=False,
        )
        x = x + self.attn_dropout(attn_out)
        x = x + self.feed_forward(self.ln2(x))
        return x


class TinyCSVLLM(nn.Module):

    def __init__(self, vocab_size, embedding_dim, num_heads,
                 num_layers, block_size, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size

        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(block_size, embedding_dim)
        self.embedding_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(embedding_dim, num_heads, dropout)
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(embedding_dim)
        self.lm_head = nn.Linear(embedding_dim, vocab_size)

        mask = torch.triu(
            torch.ones(block_size, block_size, dtype=torch.bool), diagonal=1
        )
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.block_size:
            raise ValueError(f"Sequence length {T} exceeds {self.block_size}")

        positions = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        x = self.embedding_dropout(x)

        mask = self.causal_mask[:T, :T]
        for block in self.blocks:
            x = block(x, mask)

        logits = self.lm_head(self.final_norm(x))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(B * T, self.vocab_size), targets.reshape(B * T)
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=100, temperature=0.6,
                 top_k=10, stop_ids=None, min_new_tokens=1):
        """Generate tokens, optionally halting on any id in stop_ids.

        min_new_tokens guards against stopping immediately: an empty
        record prompt ends in a newline, and the stop character is also a
        newline, so the very first sample would otherwise end generation.
        """
        was_training = self.training
        self.eval()
        stop_ids = set(stop_ids or [])

        try:
            for n in range(max_new_tokens):
                idx_context = idx[:, -self.block_size:]
                logits, _ = self(idx_context)
                logits = logits[:, -1, :] / max(temperature, 1e-6)

                if top_k is not None:
                    k = min(top_k, logits.size(-1))
                    values, _ = torch.topk(logits, k)
                    logits = logits.masked_fill(
                        logits < values[:, [-1]], float("-inf")
                    )

                probs = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
                idx = torch.cat([idx, next_token], dim=1)

                if n + 1 >= min_new_tokens and int(next_token.item()) in stop_ids:
                    break
        finally:
            if was_training:
                self.train()

        return idx


# ============================================================
# 2. LOAD CHECKPOINT
# ============================================================

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"No checkpoint at {CHECKPOINT_PATH}. Run train_csv_llm.py first."
    )

# weights_only=False: the checkpoint stores the tokenizer dicts.
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

stoi = checkpoint["stoi"]
itos = checkpoint["itos"]
block_size = checkpoint["block_size"]
row_start = checkpoint.get("row_start", "<ROW>\n")
trained_columns = checkpoint.get("csv_columns", [])
dropped_columns = checkpoint.get("dropped_columns", [])

model = TinyCSVLLM(
    vocab_size=checkpoint["vocab_size"],
    embedding_dim=checkpoint["embedding_dim"],
    num_heads=checkpoint["num_heads"],
    num_layers=checkpoint["num_layers"],
    block_size=block_size,
    dropout=checkpoint["dropout"],
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("==== LOADED ====")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Step {checkpoint.get('step', '?')} | "
      f"val loss {checkpoint.get('best_val_loss', float('nan')):.4f} | "
      f"{checkpoint['total_parameters']:,} params")
print(f"Vocabulary ({checkpoint['vocab_size']} chars): "
      f"{''.join(sorted(stoi))!r}")
print(f"\nFields the model was trained on ({len(trained_columns)}):")
print(" ", ", ".join(trained_columns))
if dropped_columns:
    print(f"\nFields dropped before training ({len(dropped_columns)}) — the "
          f"model has never seen these names:")
    print(" ", ", ".join(dropped_columns))


# ============================================================
# 3. TOKENIZER WITH COVERAGE CHECKING
# ============================================================
# The original encode() used `if ch in stoi`, which silently deleted any
# character outside the vocabulary. "qwen" became "en" with no warning.
# This version reports what is missing instead of hiding it.

def encode(text, strict=True):
    missing = sorted({ch for ch in text if ch not in stoi})

    if missing:
        shown = "".join(missing)
        message = (
            f"{len(missing)} character(s) in the prompt are not in the "
            f"model vocabulary: {shown!r}\n"
            f"    The vocabulary was built only from characters present in "
            f"the training CSVs."
        )
        if strict:
            raise ValueError(
                message + "\n    Rewrite the prompt using known characters, "
                "or pass strict=False to drop them."
            )
        print(f"\nWARNING: {message}")
        print("    Dropping them. The model will see corrupted input.")

    return [stoi[ch] for ch in text if ch in stoi]


def decode(ids):
    return "".join(itos[int(i)] for i in ids)


# ============================================================
# 4. PROMPT FORMATTING
# ============================================================
# Training rows are a single line: "<ROW>\ncol: val | col: val\n"
# A prompt with newlines around the pipes does not match that shape, so
# the model is being asked to continue a format it never saw.

def normalize_prompt(text):
    text = text.strip()

    # Strip any row marker the caller supplied, normalize the body to a
    # single line, then re-attach the marker verbatim (including its
    # trailing newline, which is part of the training format).
    marker = row_start.strip()
    if text.startswith(marker):
        text = text[len(marker):]

    text = re.sub(r"\s*\|\s*", " | ", text)   # collapse newlines around pipes
    text = re.sub(r"[ \t]+", " ", text)
    text = text.replace("\n", " ").strip()

    return row_start + text


def check_fields(text):
    """Warn about field names the model was never trained on."""
    used = re.findall(r"([A-Za-z_][A-Za-z0-9_]*)\s*:", text)
    unknown = [f for f in used if f not in trained_columns and f != "ROW"]
    if unknown:
        print(f"\nWARNING: prompt references fields absent from training: "
              f"{', '.join(sorted(set(unknown)))}")
        print("    The model cannot condition on these. Output will be noise.")
    return unknown


def run_prompt(raw_prompt, max_new_tokens=120, temperature=0.6,
               top_k=10, strict=False, min_new_tokens=1):
    prompt = normalize_prompt(raw_prompt)
    check_fields(prompt)

    prompt_ids = encode(prompt, strict=strict)
    if not prompt_ids:
        raise ValueError("Prompt contains no characters the model knows.")

    if len(prompt_ids) > block_size:
        print(f"\nNOTE: prompt is {len(prompt_ids)} tokens, context is "
              f"{block_size}. Keeping the last {block_size}.")
        prompt_ids = prompt_ids[-block_size:]

    x = torch.tensor([prompt_ids], dtype=torch.long, device=device)

    # Stop at newline: one record ends there.
    stop_ids = [stoi[c] for c in "\n" if c in stoi]

    generated = model.generate(
        x, max_new_tokens=max_new_tokens,
        temperature=temperature, top_k=top_k, stop_ids=stop_ids,
        min_new_tokens=min_new_tokens,
    )

    full = decode(generated[0].tolist())
    completion = full[len(prompt):]
    return prompt, completion


# ============================================================
# 5. YOUR ORIGINAL PROMPT
# ============================================================

original_prompt = """
model_type: qwen |
parameters: 2000000000 |
model_flops: 3000000000 |
cpu_model: Raspberry Pi 4 |
ram_total_gb: 4 |
task: predict approximate execution time |
execution_time_sec:
"""

print("\n\n" + "=" * 60)
print("ATTEMPT 1 — your prompt as written")
print("=" * 60)

prompt, completion = run_prompt(original_prompt, strict=False)
print(f"\nPrompt sent:\n  {prompt!r}")
print(f"\nCompletion:\n  {completion!r}")


# ============================================================
# 6. A PROMPT THE MODEL CAN ACTUALLY ANSWER
# ============================================================
# Built only from fields that survived training, in the exact training
# format, conditioning on the values that precede the target field in
# column order. A causal LM can only use what comes before.

def complete_field(known_values, target_field, **kwargs):
    """Build a training-shaped prefix and let the model continue it."""
    if target_field not in trained_columns:
        raise ValueError(
            f"'{target_field}' is not a trained field. Available: "
            f"{', '.join(trained_columns)}"
        )

    target_index = trained_columns.index(target_field)
    parts = []

    for col in trained_columns[:target_index]:
        if col in known_values:
            parts.append(f"{col}: {known_values[col]}")

    parts.append(f"{target_field}:")
    return run_prompt(" | ".join(parts), **kwargs)


print("\n\n" + "=" * 60)
print("ATTEMPT 2 — same intent, fields the model knows")
print("=" * 60)

known = {
    "timestamp": "2026-08-26 19:00:27",
    "sample_index": "10",
    "true_label": "3",
    "correct": "False",
    "confidence_score": "0.88",
    "logit_margin": "2.4",
    "entropy": "0.4",
}

prompt, completion = complete_field(known, "execution_time_sec")
print(f"\nPrompt sent:\n  {prompt!r}")
print(f"\nCompletion:\n  {completion!r}")


# ============================================================
# 7. FREE GENERATION FROM THE ROW MARKER
# ============================================================

print("\n\n" + "=" * 60)
print("ATTEMPT 3 — unconditional record generation")
print("=" * 60)

prompt, completion = run_prompt(
    row_start, max_new_tokens=250, temperature=0.6, min_new_tokens=40
)
print(f"\n{row_start.strip()}\n{completion}")

==== LOADED ====
Checkpoint: artifacts\csv_llm_1M_checkpoint.pth
Step 1800 | val loss 0.3562 | 1,027,275 params
Vocabulary (75 chars): '\n #()+-.0123456789:<>@ABCDEFGHIJKLMNOPQRSTUVWXY_abcdefghijklmnopqrstuvwxyz|'

Fields the model was trained on (71):
  timestamp, unique_device_id, device_short_id, pc_name, sample_index, true_label, prediction, correct, model_type, parameters, model_flops, confidence_score, logit_margin, entropy, execution_time_sec, cpu_energy_kwh, gpu_energy_kwh, ram_energy_kwh, total_energy_kwh, total_emissions_kg, carbon_intensity_kgco2_kwh, codecarbon_version, input_tokens, output_tokens, total_tokens, tokens_per_second, joules_per_token, energy_per_token_kwh, watts_estimated, gpu_energy_pct_of_total, cpu_energy_pct_of_total, cpu_model, cpu_architecture, cpu_core_count, cpu_thread_count, cpu_core, cpu_thread, cpu_usage_pct, cpu_clock_mhz, cpu_temp_c, cpu_cores_used, gpu_model, gpu_core, gpu_thread, gpu_driver_version, gpu_compute_capability, gpu_power_limit_w, gpu